# LangChain L13 — Level 12 — Multi-agent systems
As OpsPilot grows, one prompt and twenty tools become hard to steer. A common answer is to
split it into **specialists** and let a **supervisor** delegate:

```text
                    Supervisor
              (routes, combines)
                 /            \
        Billing agent      Policy agent
        get_customer       search_policies
        get_order
```

In LangChain the simplest supervisor pattern is *sub-agents as tools*: each specialist is a
`create_agent()` wrapped in a `@tool`. The supervisor sees "billing_agent" and "policy_agent"
as two capabilities and never learns their inner tools.

An engineering warning: **do not add agents because you can.** One agent with good tools is
simpler, cheaper and easier to debug. Split when there is a real boundary: different tools,
different permissions, different prompts, different owners, or different cost/latency needs.

### Step 1 — Two specialists, each wrapped as a tool

In [ ]:
billing_specialist = create_agent(model=model, tools=[get_customer, get_order],   # LangChain
                                  system_prompt="You are the billing desk. Look up customers and orders and report the facts with ids. Never speculate about policy.")
policy_specialist = create_agent(model=model, tools=[search_policies],
                                 system_prompt="You are the policy desk. Answer only from policy documents and cite the source file.")

SPECIALIST_MODEL_CALLS = []          # ours: how many model calls each delegated task cost (for Step 3)

@tool                                # LangChain decorator: the whole specialist becomes one tool
def billing_agent(query: str) -> str:
    """Delegate to the billing desk: customer records, orders, charges. Give it a complete, self-contained question."""
    out = billing_specialist.invoke({"messages": [{"role": "user", "content": query}]})
    SPECIALIST_MODEL_CALLS.append(sum(1 for m in out["messages"] if isinstance(m, AIMessage)))
    return text_of(out["messages"][-1])

@tool
def policy_agent(query: str) -> str:
    """Delegate to the policy desk: refund, shipping and escalation rules. Give it a complete, self-contained question."""
    out = policy_specialist.invoke({"messages": [{"role": "user", "content": query}]})
    SPECIALIST_MODEL_CALLS.append(sum(1 for m in out["messages"] if isinstance(m, AIMessage)))
    return text_of(out["messages"][-1])

print("specialists as tools:", [billing_agent.name, policy_agent.name])

### Step 2 — The supervisor delegates and combines

The question needs both desks. Watch the supervisor's trajectory: it calls both specialists,
then writes one answer. The specialists' own tool calls happen inside their tools and are
invisible to the supervisor, which is the point of the boundary.

In [ ]:
supervisor = create_agent(model=model, tools=[billing_agent, policy_agent],   # LangChain: specialists are just tools here
                          system_prompt="You are OpsPilot's supervisor. Delegate to the billing and policy desks as needed, then answer the user in one short reply.")

out = supervisor.invoke({"messages": [{"role": "user", "content": "I am customer C002 and I was charged twice for order O1002. Check the order and tell me what the refund policy says."}]})
print("SUPERVISOR TRAJECTORY:")
show_messages(out["messages"])

### Step 3 — When one agent is better

The same question answered by a single agent that owns all the tools. Compare the number of
model calls: the supervisor pattern paid for three agents' worth of reasoning. Choose the
specialist split only when the boundary buys you something (permissions, prompts, ownership).

In [ ]:
single = create_agent(model=model, tools=[get_customer, get_order, search_policies], system_prompt=OPSPILOT_PROMPT)   # LangChain
question = "I am customer C002 and I was charged twice for order O1002. Check the order and tell me what the refund policy says."

def count_model_calls(messages):                                # ours: one AIMessage = one model call
    return sum(1 for m in messages if isinstance(m, AIMessage))

SPECIALIST_MODEL_CALLS.clear()
supervisor_out = supervisor.invoke({"messages": [{"role": "user", "content": question}]})
single_out = single.invoke({"messages": [{"role": "user", "content": question}]})
print("single agent      : model calls =", count_model_calls(single_out["messages"]))
print("supervisor pattern: model calls =", count_model_calls(supervisor_out["messages"]), "(supervisor) +", sum(SPECIALIST_MODEL_CALLS), "(specialists) =",
      count_model_calls(supervisor_out["messages"]) + sum(SPECIALIST_MODEL_CALLS))

### Recap

- **Problem seen:** one prompt with every tool becomes hard to steer and impossible to permission separately.
- **Layer added:** specialists built with `create_agent()` and exposed to a supervisor as tools.
- **Evidence:** the supervisor delegated to both desks and combined them; the single agent did the same job with fewer total model calls.